In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_151140"


# 🌍 Notebook 12: Coleta Multi-Fonte de Notícias (2018-2025)

**Objetivo:** Construir base histórica robusta de notícias em português sobre Ibovespa/mercado financeiro brasileiro.

**Estratégia de Coleta:**

1. **GDELT** (fonte primária histórica)
   - Cobertura: 2015-presente (usaremos 2018-2025)
   - Limitação: Não retorna texto completo (apenas título + metadados)
   - Vantagem: Gratuito, sem limite de histórico, atualização contínua
   - Taxa: ~50 req/min (com delay de 1.5s entre chamadas)

2. **NewsAPI** (fonte complementar recente)
   - Cobertura: Últimos 30 dias (plano FREE)
   - Vantagem: Alta qualidade, descrição + snippet de conteúdo
   - Limitação: Histórico limitado a 30 dias no plano gratuito
   - Taxa: 100 req/dia

**Schema Unificado:**
- `date`: datetime normalizado (meia-noite BRT)
- `source`: nome do veículo/domínio
- `title`: título da notícia
- `url`: link do artigo
- `text_full`: texto disponível (título + descrição/conteúdo)

**Resultado Final:**
- `data_raw/news_multisource.parquet`: Base bruta consolidada
- Cobertura esperada: ~2018-2025 (dentro das limitações das APIs)

---

In [2]:
# 1. Setup de caminhos e constantes
import os
import sys
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# Adiciona src ao path para imports
sys.path.insert(0, str(Path(".").absolute().parent))

from src.io import paths
from src.config import loader as cfg
from src.config.constants import START_DATE, END_DATE, START_DATE_STR, END_DATE_STR

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "12_data_collection_multisource"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("="*80)
print(f"NOTEBOOK: {NB_NAME}")
print(f"EXECUÇÃO: {STAMP}")
print("="*80)
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"RAW_PATH: {RAW_PATH}")
print(f"\n📅 PERÍODO ALVO (constants.py):")
print(f"   START_DATE: {START_DATE} ({START_DATE_STR})")
print(f"   END_DATE: {END_DATE} ({END_DATE_STR})")
print(f"   Dias teóricos: {(END_DATE - START_DATE).days + 1}")
print("="*80)

NOTEBOOK: 12_data_collection_multisource
EXECUÇÃO: 20251126_151143

BASE_PATH: C:\TCC_USP
RAW_PATH: C:\TCC_USP\data_raw

📅 PERÍODO ALVO (constants.py):
   START_DATE: 2018-01-02 (2018-01-02)
   END_DATE: 2025-12-31 (2025-12-31)
   Dias teóricos: 2921


In [3]:
# 2. Importações e carregamento dos coletores
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import importlib
import sys

# Forçar reload dos módulos (garantir versão atualizada)
if 'src.utils.gdelt_collector' in sys.modules:
    importlib.reload(sys.modules['src.utils.gdelt_collector'])
if 'src.utils.newsapi_collector' in sys.modules:
    importlib.reload(sys.modules['src.utils.newsapi_collector'])

# Import dos coletores customizados
from src.utils.gdelt_collector import GDELTCollector, collect_gdelt_historical
from src.utils.newsapi_collector import NewsAPICollector, collect_newsapi_recent

print("✅ Coletores importados com sucesso (reloaded)")
print("   - GDELTCollector: Coleta histórica 2015-presente")
print("   - NewsAPICollector: Coleta últimos 30 dias (plano FREE)")

✅ Coletores importados com sucesso (reloaded)
   - GDELTCollector: Coleta histórica 2015-presente
   - NewsAPICollector: Coleta últimos 30 dias (plano FREE)


In [4]:
# 3. Configuração da coleta

# MODO DE COLETA (altere conforme necessário)
# "FULL": Coleta histórica GDELT completa (START_DATE → END_DATE) - RECOMENDADO PARA TCC
# "RECENT": Apenas NewsAPI últimos 30 dias (teste rápido, NÃO usar para TCC oficial)
# "TEST": Teste com 7 dias GDELT (validação rápida)

COLLECT_MODE = "FULL"  # ← MODO DEFINITIVO PARA TCC

# Limiar mínimo de dias distintos (checagem obrigatória)
MIN_DAYS_THRESHOLD = 200  # Base com menos de 200 dias é insuficiente para TCC

print("="*80)
print(f"MODO DE COLETA: {COLLECT_MODE}")
print(f"LIMIAR MÍNIMO: {MIN_DAYS_THRESHOLD} dias distintos")
print("="*80)

if COLLECT_MODE == "FULL":
    print("🌍 COLETA HISTÓRICA COMPLETA (GDELT)")
    print(f"   Período: {START_DATE_STR} → {END_DATE_STR}")
    print(f"   Query: Termos financeiros PT-BR (Ibovespa, B3, Bolsa, ações)")
    print(f"   Tempo estimado: 6-12 horas (depende de conexão e API)")
    print(f"   Resultado esperado: ~10.000-50.000 artigos (vários anos)")
    print(f"\n✅ MODO OFICIAL PARA TCC - Base histórica robusta")
    print(f"⚠️ Checagem: erro fatal se < {MIN_DAYS_THRESHOLD} dias")
elif COLLECT_MODE == "RECENT":
    print("⚠️ MODO TESTE (NewsAPI FREE - NÃO USAR PARA TCC)")
    print(f"   Período: Últimos 30 dias (limitação API FREE)")
    print(f"   Cobertura: ~1-2 dias reais (limitação conhecida)")
    print(f"\n❌ NÃO ADEQUADO PARA TCC - Base insuficiente")
elif COLLECT_MODE == "TEST":
    print("🧪 MODO TESTE (GDELT 7 dias)")
    print(f"   Período: Últimos 7 dias")
    print(f"   Tempo estimado: 1-2 minutos")
    print(f"\n⚠️ Apenas validação - NÃO usar para TCC")
    
print("="*80)

MODO DE COLETA: FULL
LIMIAR MÍNIMO: 200 dias distintos
🌍 COLETA HISTÓRICA COMPLETA (GDELT)
   Período: 2018-01-02 → 2025-12-31
   Query: Termos financeiros PT-BR (Ibovespa, B3, Bolsa, ações)
   Tempo estimado: 6-12 horas (depende de conexão e API)
   Resultado esperado: ~10.000-50.000 artigos (vários anos)

✅ MODO OFICIAL PARA TCC - Base histórica robusta
⚠️ Checagem: erro fatal se < 200 dias


In [5]:
# 4. Coleta via GDELT (fonte primária histórica)

print("\n" + "="*80)
print("ETAPA 1: COLETA GDELT (Base Histórica TCC)")
print("="*80 + "\n")

if COLLECT_MODE == "FULL":
    # COLETA HISTÓRICA COMPLETA (MODO OFICIAL TCC)
    print("🌍 Iniciando coleta GDELT histórica completa...")
    print(f"⏱️ Tempo estimado: 6-12 horas para {(END_DATE - START_DATE).days} dias")
    print("💾 Checkpoints automáticos a cada 30 dias\n")
    
    try:
        df_gdelt = collect_gdelt_historical(
            start_date=START_DATE_STR,
            end_date=END_DATE_STR,
            query="(Ibovespa OR Bovespa OR B3 OR 'Bolsa de valores' OR ações OR mercado) sourcelang:por",
            output_path=Path(RAW_PATH) / "gdelt_historical.parquet",
            checkpoint_interval=30,
            min_days_threshold=MIN_DAYS_THRESHOLD,
        )
        
        print(f"\n✅ COLETA GDELT CONCLUÍDA COM SUCESSO")
        print(f"   Base aprovada: {df_gdelt['date'].nunique():,} dias (>= {MIN_DAYS_THRESHOLD})")
        
    except RuntimeError as e:
        print(f"\n❌ ERRO FATAL NA COLETA GDELT:")
        print(f"   {str(e)}")
        raise  # Interrompe execução - base insuficiente
    
elif COLLECT_MODE == "TEST":
    # Teste rápido: últimos 7 dias
    print("🧪 Teste GDELT: últimos 7 dias (validação)")
    collector = GDELTCollector(rate_limit_delay=0.5)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=7)
    
    df_gdelt = collector.collect_by_date_range(
        start_date=start_date,
        end_date=end_date,
        query="(Ibovespa OR Bovespa OR bolsa OR ações OR mercado) sourcelang:por",
        max_records=250,
    )
    
    n_days_test = df_gdelt['date'].nunique() if not df_gdelt.empty else 0
    print(f"\n⚠️ MODO TESTE: {n_days_test} dias coletados")
    print(f"   (Limiar {MIN_DAYS_THRESHOLD} dias NÃO aplicado em TEST)")
    
else:  # RECENT mode
    print("⏭️ Modo RECENT: pulando coleta GDELT")
    print("⚠️ NewsAPI FREE sozinho NÃO é adequado para TCC")
    df_gdelt = pd.DataFrame()

# Resumo GDELT
if not df_gdelt.empty:
    print("\n📊 Resumo GDELT:")
    print(f"   Total: {len(df_gdelt):,} artigos")
    print(f"   Período: {df_gdelt['date'].min().date()} → {df_gdelt['date'].max().date()}")
    print(f"   Dias distintos: {df_gdelt['date'].nunique():,}")
    print(f"   Fontes únicas: {df_gdelt['source'].nunique():,}")
    display(df_gdelt.head(3))
else:
    print("⚠️ GDELT: Nenhum dado coletado")


ETAPA 1: COLETA GDELT (Base Histórica TCC)

🌍 Iniciando coleta GDELT histórica completa...
⏱️ Tempo estimado: 6-12 horas para 2920 dias
💾 Checkpoints automáticos a cada 30 dias


COLETA HISTÓRICA GDELT: 2018-01-02 → 2025-12-31
Query: (Ibovespa OR Bovespa OR B3 OR 'Bolsa de valores' OR ações OR...
Checagem mínima: 200 dias distintos

🌍 GDELT: Coletando 2018-01-02 → 2018-01-31


  ✅ 2018-01-02: 78 artigos


  ✅ 2018-01-03: 117 artigos


  ✅ 2018-01-04: 75 artigos


  ✅ 2018-01-05: 79 artigos


...

  ✅ 2019-04-29: 59 artigos


  ✅ 2019-04-30: 48 artigos


  ✅ 2019-05-01: 15 artigos


  ✅ 2019-05-02: 50 artigos


  ✅ 2019-05-03: 65 artigos


In [ ]:
# Verificação das estatísticas da coleta GDELT
print("\n" + "="*80)
print("VERIFICAÇÃO DA COLETA GDELT")
print("="*80)
print(f"Total de artigos: {len(df_gdelt):,}")
print(f"Dias distintos: {df_gdelt['date'].nunique():,}")
print(f"Período: {df_gdelt['date'].min()} → {df_gdelt['date'].max()}")
print(f"Fontes únicas: {df_gdelt['source'].nunique():,}")
print(f"Média diária: {len(df_gdelt) / df_gdelt['date'].nunique():.1f} artigos/dia")
print("="*80)

In [ ]:
# 5. Coleta via NewsAPI (complementar - NÃO usado em modo FULL)

print("\n" + "="*80)
print("ETAPA 2: NewsAPI (Complementar)")
print("="*80 + "\n")

if COLLECT_MODE == "FULL":
    print("⏭️ Modo FULL: NewsAPI desabilitado")
    print("   Base oficial TCC vem 100% do GDELT histórico")
    print("   (NewsAPI FREE tem limitações de histórico)")
    df_newsapi = pd.DataFrame()
    
elif COLLECT_MODE == "RECENT":
    print("📰 Coletando NewsAPI: últimos 30 dias (teste)")
    
    # Chave NewsAPI
    NEWSAPI_KEY = "3d606b02f24447849f49b3e8c3628f46"
    
    collector_newsapi = NewsAPICollector(
        api_key=NEWSAPI_KEY,
        rate_limit_delay=1.0,
    )
    
    df_newsapi = collector_newsapi.collect_recent(days=30)
    
    if not df_newsapi.empty:
        print(f"\n📊 Resumo NewsAPI:")
        print(f"   Total: {len(df_newsapi):,} artigos")
        print(f"   Período: {df_newsapi['date'].min()} → {df_newsapi['date'].max()}")
        print(f"   Dias distintos: {df_newsapi['date'].nunique():,}")
        display(df_newsapi.head(3))
    else:
        print("⚠️ NewsAPI: Nenhum dado retornado")
        
elif COLLECT_MODE == "TEST":
    print("⏭️ Modo TEST: NewsAPI desabilitado (apenas GDELT)")
    df_newsapi = pd.DataFrame()
else:
    df_newsapi = pd.DataFrame()

In [ ]:
# 6. Consolidação e salvamento final

print("\n" + "="*80)
print("ETAPA 3: CONSOLIDAÇÃO E SALVAMENTO")
print("="*80 + "\n")

# Consolidar fontes
dataframes = []

if not df_gdelt.empty:
    print(f"✅ GDELT: {len(df_gdelt):,} artigos")
    dataframes.append(df_gdelt)

if not df_newsapi.empty:
    print(f"✅ NewsAPI: {len(df_newsapi):,} artigos")
    dataframes.append(df_newsapi)

if not dataframes:
    raise RuntimeError(
        "[COLETA] ERRO CRÍTICO: Nenhuma fonte retornou dados!\n"
        "   Verifique: conexão, API keys, rate limits, query GDELT"
    )

# Concatenar e deduplicar
df_consolidated = pd.concat(dataframes, ignore_index=True)

print(f"\n📊 Antes da deduplicação: {len(df_consolidated):,} artigos")

# Deduplicação por URL (primário) e título (fallback)
df_consolidated = df_consolidated.drop_duplicates(subset=["url"])
df_consolidated = df_consolidated.sort_values("date").reset_index(drop=True)

print(f"📊 Após deduplicação: {len(df_consolidated):,} artigos")

# Validação de qualidade
df_consolidated = df_consolidated.dropna(subset=["date", "title"])
df_consolidated = df_consolidated[df_consolidated["title"].str.len() >= 20]

print(f"📊 Após filtros de qualidade: {len(df_consolidated):,} artigos\n")

# ========================================================================
# CHECAGEM OBRIGATÓRIA: Mínimo de dias distintos
# ========================================================================
n_days_final = df_consolidated["date"].nunique()
n_news_final = len(df_consolidated)

print("="*80)
print("ESTATÍSTICAS FINAIS")
print("="*80)
print(f"Total de artigos: {n_news_final:,}")
print(f"Período coberto: {df_consolidated['date'].min().date()} → {df_consolidated['date'].max().date()}")
print(f"Dias distintos: {n_days_final:,}")
print(f"Fontes únicas: {df_consolidated['source'].nunique():,}")
print(f"Média diária: {n_news_final / n_days_final:.1f} artigos/dia")
print("="*80 + "\n")

# VALIDAÇÃO DE LIMIAR
if n_days_final < MIN_DAYS_THRESHOLD:
    raise RuntimeError(
        f"[COLETA] Base de notícias INSUFICIENTE para TCC:\n"
        f"   Dias coletados: {n_days_final}\n"
        f"   Mínimo exigido: {MIN_DAYS_THRESHOLD}\n"
        f"   AÇÃO: Revisar modo de coleta, período ou fontes"
    )

print(f"✅ VALIDAÇÃO APROVADA: {n_days_final:,} dias >= {MIN_DAYS_THRESHOLD} (limiar mínimo)\n")

# Top 10 fontes
print("📰 Top 10 fontes mais frequentes:")
print(df_consolidated["source"].value_counts().head(10))

# Amostra
print("\n📄 Amostra dos dados:")
display(df_consolidated.sample(min(5, len(df_consolidated))))

# ========================================================================
# SALVAMENTO: data_raw/news_multisource.parquet (BASE OFICIAL TCC)
# ========================================================================
output_file = Path(RAW_PATH) / "news_multisource.parquet"
df_consolidated.to_parquet(output_file, index=False)

print(f"\n💾 BASE OFICIAL TCC salva em:")
print(f"   {output_file}")
print(f"   Tamanho: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

# Backup CSV (para inspeção manual se necessário)
output_csv = Path(RAW_PATH) / "news_multisource.csv"
df_consolidated.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"   Backup CSV: {output_csv}")

print("\n" + "="*80)
print("✅ COLETA CONCLUÍDA COM SUCESSO!")
print(f"   Arquivo: news_multisource.parquet")
print(f"   Registros: {n_news_final:,}")
print(f"   Dias: {n_days_final:,}")
print(f"   Status: APROVADO para TCC (>= {MIN_DAYS_THRESHOLD} dias)")
print("="*80)